# 05 · Reset de Tablas
Borra el contenido de las tablas que elijas y las deja vacías, listas para volver a cargar con el notebook `03`.

> ⚠️ **Esta operación es irreversible.** Asegúrate de qué tablas quieres resetear antes de ejecutar.

In [ ]:
from google.cloud import bigquery
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
DATASET_ID = os.getenv('BQ_DATASET_ID')
client = bigquery.Client(project=PROJECT_ID)
DS = f'{PROJECT_ID}.{DATASET_ID}'
print('Conexión OK ✓')

In [ ]:
# ── Elige qué tablas resetear ──────────────────────────────────
# Pon True en las que quieras vaciar, False en las que no tocar

TABLAS_A_RESETEAR = {
    'categories':    False,
    'customers':     False,
    'products':      False,
    'orders':        False,
    'order_item_id': False,
    'payments':      False,
    'reviews':       True,   # <-- ejemplo: solo resetear reviews
}

In [ ]:
# ── Previsualización: qué se va a borrar ──────────────────────
print('Tablas seleccionadas para reset:\n')
for tabla, activa in TABLAS_A_RESETEAR.items():
    if activa:
        n = client.query(f'SELECT COUNT(*) as n FROM `{DS}.{tabla}`').result().to_dataframe().iloc[0]['n']
        print(f'  🗑️  {tabla:<22} → {n} filas se borrarán')

print('\nLas demás tablas NO se tocarán.')

In [ ]:
# ── Ejecutar el reset ──────────────────────────────────────────
# TRUNCATE es más rápido que DELETE y no consume slots de DML

for tabla, activa in TABLAS_A_RESETEAR.items():
    if not activa:
        print(f'  ⏭️  {tabla:<22} omitida')
        continue
    job = client.query(f'TRUNCATE TABLE `{DS}.{tabla}`')
    job.result()
    print(f'  ✓  {tabla:<22} vaciada')

print('\nReset completado. Ahora puedes ejecutar el notebook 03 para recargar los datos.')

In [ ]:
# ── Verificación final ─────────────────────────────────────────
import pandas as pd

todas = ['categories','customers','products','orders','order_item_id','payments','reviews']
rows  = []
for t in todas:
    n = client.query(f'SELECT COUNT(*) as n FROM `{DS}.{t}`').result().to_dataframe().iloc[0]['n']
    rows.append({'tabla': t, 'filas': n})

pd.DataFrame(rows)